# Day 2 - Beginner track
## Training models properly, and the metrics that tell you whether it worked

**MIPT International Autumn School 2026 | 9 September | 90 minutes**

Yesterday you cleaned the data. Today you will train models on it, and more importantly,
you will learn to tell the difference between a model that works and one that only looks
like it works.

### What is different from yesterday

Yesterday we filled missing values using the median of the whole dataset, and the notebook
flagged that as technically cheating. Today you will do it correctly, using a scikit-learn
`Pipeline`. That is not a formality: it is the machinery that makes the difference between
an honest estimate of performance and a flattering one.

### Core sections

Parts 1 to 5 are the core. Parts 6 and 7 are the interesting part, so protect time for them.
Everything after that is bonus.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

import sklearn
print("scikit-learn:", sklearn.__version__)

SEED = 42
np.random.seed(SEED)

---
# Part 1 - Cleaning that is safe to do before splitting

There are two kinds of cleaning, and the distinction matters enormously.

**Deterministic fixes** depend only on the row in front of you. Stripping whitespace from
`' Toyota '`, turning `-1` into "unknown", removing duplicate rows. These use no statistics
computed across the dataset, so doing them before the split leaks nothing.

**Statistical fixes** depend on the whole column. Filling a gap with the median, scaling by
the standard deviation, choosing which categories exist. These must be computed on the
training data only, or information about your test set flows into your model.

Do the first kind now. The second kind goes inside a `Pipeline` in Part 3.

In [ ]:
DATA_PATH = "data/car_listings.csv"
raw = pd.read_csv(DATA_PATH)

df = raw.copy()

# deterministic fix 1: whitespace and inconsistent capitalisation
for col in ["brand", "fuel_type", "transmission", "body_type", "city", "service_history"]:
    df[col] = df[col].str.strip()
df["fuel_type"] = df["fuel_type"].str.lower()
df["city"] = df["city"].str.lower()

# deterministic fix 2: sentinel values become honest missing values
df.loc[df["mileage_km"] < 0, "mileage_km"] = np.nan
df.loc[df["model_year"] < 1990, "model_year"] = np.nan

# deterministic fix 3: the ten-times data entry errors
big = df["mileage_km"] > 500_000
df.loc[big, "mileage_km"] = df.loc[big, "mileage_km"] / 10

# deterministic fix 4: duplicate records
df = df.drop_duplicates(subset=[c for c in df.columns if c != "listing_id"]).reset_index(drop=True)

# a derived feature, computed per row, so also safe
df["age_years"] = 2026 - df["model_year"]

print("rows after deterministic cleaning:", len(df))
print("remaining missing values:")
print(df.isna().sum()[lambda s: s > 0])

Notice what we did **not** do: we did not fill a single missing value. Those gaps are still
there, and they will be filled inside the pipeline, using training-set statistics only.

---
# Part 2 - Split first

Three splits, and each has exactly one job.

In [ ]:
from sklearn.model_selection import train_test_split

NUMERIC = ["age_years", "mileage_km", "engine_litres", "owners", "accident_reported"]
CATEGORICAL = ["brand", "fuel_type", "transmission", "body_type", "city", "service_history"]
FEATURES = NUMERIC + CATEGORICAL

X = df[FEATURES]
y_price = df["price_eur"]
y_sold = df["sold_fast"]

# 70 / 15 / 15. Two calls, because train_test_split only makes two pieces at a time.
X_tr, X_tmp, yp_tr, yp_tmp, ys_tr, ys_tmp = train_test_split(
    X, y_price, y_sold, test_size=0.30, random_state=SEED)
X_va, X_te, yp_va, yp_te, ys_va, ys_te = train_test_split(
    X_tmp, yp_tmp, ys_tmp, test_size=0.50, random_state=SEED)

print(f"train      {len(X_tr):5d}   the model learns from this")
print(f"validation {len(X_va):5d}   you choose between models with this")
print(f"test       {len(X_te):5d}   touched once, at the very end")

---
# Part 3 - The pipeline: doing preprocessing honestly

A `Pipeline` chains preprocessing steps and a model into one object. When you call `.fit()`
on it, every step learns its parameters from the training data. When you call `.predict()`,
those same learned parameters are applied to the new data without being recomputed.

That single property is what prevents the leak we committed yesterday.

A `ColumnTransformer` lets different columns take different routes: numbers get imputed and
scaled, text gets imputed and one-hot encoded.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_steps = Pipeline([
    ("impute", SimpleImputer(strategy="median")),     # median learned from TRAIN only
    ("scale", StandardScaler()),                      # mean and sd learned from TRAIN only
])

categorical_steps = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value="unknown")),
    ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocess = ColumnTransformer([
    ("num", numeric_steps, NUMERIC),
    ("cat", categorical_steps, CATEGORICAL),
])

# See what it produces
preprocess.fit(X_tr)
print("columns after preprocessing:", preprocess.transform(X_tr).shape[1])
print("\nfirst few generated column names:")
print(list(preprocess.get_feature_names_out())[:8])

`handle_unknown="ignore"` matters. If a brand appears in the test set but never in the
training set, the encoder produces all zeros for it instead of crashing. In production this
happens constantly, and a model that crashes on an unseen category is a model that stops
working the first time a new car brand launches.

---
# Part 4 - Regression, and the baseline you must always beat

Before any real model, fit the stupidest thing that could possibly work. If your
sophisticated model cannot beat "always predict the average", it is not a model, it is a
random number generator with extra steps.

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score


def regression_report(name, model, log_target=False):
    """Fit on train, score on validation, return a row of metrics."""
    target = np.log(yp_tr) if log_target else yp_tr
    model.fit(X_tr, target)
    pred = model.predict(X_va)
    if log_target:
        pred = np.exp(pred)
    return {
        "model": name,
        "MAE (EUR)": int(mean_absolute_error(yp_va, pred)),
        "RMSE (EUR)": int(root_mean_squared_error(yp_va, pred)),
        "R2": round(r2_score(yp_va, pred), 3),
        "MAPE (%)": round(float(np.mean(np.abs(pred - yp_va) / yp_va) * 100), 1),
    }


rows = []
rows.append(regression_report("always predict the mean",
                              Pipeline([("prep", preprocess), ("model", DummyRegressor(strategy="mean"))])))
rows.append(regression_report("linear regression",
                              Pipeline([("prep", preprocess), ("model", LinearRegression())])))

pd.DataFrame(rows).set_index("model")

### The four metrics, and when each one lies to you

| Metric | What it is | When it misleads |
|---|---|---|
| **MAE** | average size of the error, in euros | treats a 500 EUR miss on a cheap car as equal to one on an expensive car |
| **RMSE** | like MAE but squares the errors first | a single catastrophic prediction dominates it |
| **R squared** | fraction of the variance explained; 0 means no better than the mean | looks impressive on easy problems, and can be negative |
| **MAPE** | average error as a percentage of the true value | explodes when true values approach zero |

Report at least two, and always report one in the units the audience thinks in. Nobody
outside this room knows what an R squared of 0.8 feels like. Everybody understands
"typically wrong by about 2,200 euros".

### YOUR TURN 1

Add the log-target version of linear regression to the table. The `regression_report`
function already supports it with `log_target=True`.

Before you run it, write down what you expect to happen and why. You saw the histogram
yesterday.

In [ ]:
# TODO: your prediction, as a comment


# TODO: append the log-target linear regression to `rows`


pd.DataFrame(rows).set_index("model")

**Checkpoint.** Log-target linear regression should roughly halve the MAE and take MAPE from
around 20 percent down to around 10. Nothing about the algorithm changed. Only the target did.

This is the most important practical lesson of the week: understanding your problem beats
reaching for a bigger model.

---
# Part 5 - Trees, forests, and what overfitting looks like

A decision tree splits the data repeatedly on feature thresholds. Left unconstrained, it will
keep splitting until every training row sits in its own leaf, at which point it has memorised
the training set perfectly and learned nothing.

That is overfitting, and this is what it looks like.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

depths = range(1, 21)
train_scores, val_scores = [], []

for d in depths:
    m = Pipeline([("prep", preprocess),
                  ("model", DecisionTreeRegressor(max_depth=d, random_state=SEED))])
    m.fit(X_tr, np.log(yp_tr))
    train_scores.append(r2_score(np.log(yp_tr), m.predict(X_tr)))
    val_scores.append(r2_score(np.log(yp_va), m.predict(X_va)))

plt.plot(depths, train_scores, label="training set", color="#B85042", marker="o", ms=4)
plt.plot(depths, val_scores, label="validation set", color="#50808E", marker="o", ms=4)
plt.xlabel("max_depth"); plt.ylabel("R squared on log(price)")
plt.title("A decision tree overfitting in real time")
plt.xticks(list(depths)[::2]); plt.legend(); plt.show()

best = depths[int(np.argmax(val_scores))]
print(f"best depth on validation: {best}  (val R2 = {max(val_scores):.3f})")
print(f"at depth 20: train R2 = {train_scores[-1]:.3f}, val R2 = {val_scores[-1]:.3f}")

Read that plot carefully, because you will see its shape for the rest of your career.

* The training curve rises forever. It always will. A model can always memorise more.
* The validation curve rises, peaks, then falls. The peak is the useful model.
* The gap between the curves is how much the model has memorised rather than learned.

**The training score is not evidence of anything.** It is the model's opinion of its own
homework.

### YOUR TURN 2

Fit a `RandomForestRegressor` on the log target and add it to the results table.

A random forest trains many trees, each on a random subsample of rows and columns, and
averages them. The individual trees still overfit, but their errors are partly independent,
so averaging cancels much of the damage.

Use `n_estimators=300, min_samples_leaf=2, random_state=SEED, n_jobs=-1`.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# TODO: build the pipeline, report it, and append to `rows`


pd.DataFrame(rows).set_index("model")

Compare the random forest against log-target linear regression. On this dataset they are
close, and the linear model may well win.

That is worth sitting with. The random forest has three hundred trees and takes a hundred
times longer to train. The linear model is a set of coefficients you could write on a napkin,
and you can read straight off it how much an accident costs a seller.

**When two models perform the same, choose the one you can explain.**

---
# Part 6 - Cross-validation

A single validation split of 600 rows gives a noisy estimate. Change the random seed and your
"best" model can change with it.

K-fold cross-validation splits the training data into k parts, trains k times, and each time
holds out a different part. You get k scores instead of one, so you can see the spread as
well as the average.

In [ ]:
from sklearn.model_selection import cross_val_score, KFold

kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

candidates = {
    "linear regression": LinearRegression(),
    "decision tree (depth 8)": DecisionTreeRegressor(max_depth=8, random_state=SEED),
    "random forest": RandomForestRegressor(n_estimators=200, min_samples_leaf=2,
                                           random_state=SEED, n_jobs=-1),
}

for name, model in candidates.items():
    pipe = Pipeline([("prep", preprocess), ("model", model)])
    scores = cross_val_score(pipe, X_tr, np.log(yp_tr), cv=kf, scoring="r2")
    print(f"{name:26s} R2 = {scores.mean():.3f}  +/- {scores.std():.3f}   folds: {np.round(scores, 3)}")

Look at the spread, not only the mean. If two models differ by 0.004 and the fold-to-fold
standard deviation is 0.02, you have not shown that one is better. You have shown that your
measurement is too noisy to tell.

Reporting a difference smaller than your own error bars is one of the most common failures in
student projects, and in a fair number of published papers.

---
# Part 7 - Classification, where the metrics get interesting

Switch targets. `sold_fast` is 1 if the listing sold within 30 days.

Accuracy is the obvious metric and the one that misleads most often. Start with the baseline.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             ConfusionMatrixDisplay, RocCurveDisplay)

print("class balance in training data:", np.round(ys_tr.value_counts(normalize=True).sort_index().values, 3))


def classification_report_row(name, model):
    model.fit(X_tr, ys_tr)
    pred = model.predict(X_va)
    proba = model.predict_proba(X_va)[:, 1]
    return {
        "model": name,
        "accuracy": round(accuracy_score(ys_va, pred), 3),
        "precision": round(precision_score(ys_va, pred, zero_division=0), 3),
        "recall": round(recall_score(ys_va, pred, zero_division=0), 3),
        "F1": round(f1_score(ys_va, pred, zero_division=0), 3),
        "ROC-AUC": round(roc_auc_score(ys_va, proba), 3),
    }


crows = []
crows.append(classification_report_row("always predict the majority",
    Pipeline([("prep", preprocess), ("model", DummyClassifier(strategy="most_frequent"))])))
crows.append(classification_report_row("logistic regression",
    Pipeline([("prep", preprocess), ("model", LogisticRegression(max_iter=2000))])))
crows.append(classification_report_row("random forest",
    Pipeline([("prep", preprocess), ("model", RandomForestClassifier(
        n_estimators=300, min_samples_leaf=3, random_state=SEED, n_jobs=-1))])))

pd.DataFrame(crows).set_index("model")

### The four classification metrics, in plain language

Your model predicts "will sell fast". Sometimes it is right, sometimes wrong, and the two
kinds of wrong are not the same.

| | |
|---|---|
| **Precision** | of the listings you predicted would sell fast, what fraction actually did? |
| **Recall** | of the listings that actually sold fast, what fraction did you catch? |
| **F1** | the harmonic mean of the two, for when you need a single number |
| **ROC-AUC** | how well the model ranks; 0.5 is random, 1.0 is perfect. Ignores the threshold entirely |

Precision and recall trade against each other. Predict "fast" for everything and recall hits
1.0 while precision collapses. Predict "fast" only for the single listing you are most
certain about and precision is 1.0 while recall is near zero.

**Which one matters is a business question, not a modelling one.** Ask what the wrong answer
costs in each direction.

In [ ]:
# The confusion matrix shows all four outcomes at once. Learn to read it.
best_clf = Pipeline([("prep", preprocess), ("model", LogisticRegression(max_iter=2000))])
best_clf.fit(X_tr, ys_tr)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
ConfusionMatrixDisplay.from_estimator(
    best_clf, X_va, ys_va, ax=axes[0], cmap="OrRd", colorbar=False,
    display_labels=["slow", "fast"])
axes[0].set_title("Confusion matrix")
axes[0].grid(False)

RocCurveDisplay.from_estimator(best_clf, X_va, ys_va, ax=axes[1],
                               curve_kwargs={"color": "#B85042"})
axes[1].plot([0, 1], [0, 1], "--", color="#999999", lw=1)
axes[1].set_title("ROC curve")
plt.tight_layout(); plt.show()

cm = confusion_matrix(ys_va, best_clf.predict(X_va))
print("true negatives ", cm[0, 0], "  correctly called slow")
print("false positives", cm[0, 1], "  called fast, actually slow")
print("false negatives", cm[1, 0], "  called slow, actually fast")
print("true positives ", cm[1, 1], "  correctly called fast")

### YOUR TURN 3

The default threshold of 0.5 is a convention, not a law. Moving it trades precision against
recall.

Imagine the business context: you are advising a dealer on which cars to buy at auction.
A listing you wrongly call "fast" ties up capital in stock that does not move. A listing you
wrongly call "slow" is only a missed opportunity.

That asymmetry means you want **high precision**, even at the cost of recall.

Sweep the threshold from 0.1 to 0.9, plot precision and recall against it, and pick a
threshold that gets precision to at least 0.80. Report the recall you gave up to get there.

In [ ]:
proba = best_clf.predict_proba(X_va)[:, 1]
thresholds = np.arange(0.10, 0.91, 0.02)

# TODO: compute precision and recall at each threshold
precisions, recalls = [], []


# TODO: plot both against the threshold


# TODO: report the first threshold reaching precision >= 0.80, and its recall

---
# Part 8 - The trap, sprung

Yesterday's stretch task pointed you at `days_on_market`. Here is what happens when you use it.

In [ ]:
LEAKY = FEATURES + ["days_on_market"]

leaky_preprocess = ColumnTransformer([
    ("num", numeric_steps, NUMERIC + ["days_on_market"]),
    ("cat", categorical_steps, CATEGORICAL),
])

leaky = Pipeline([("prep", leaky_preprocess),
                  ("model", RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1))])
leaky.fit(df.loc[X_tr.index, LEAKY], ys_tr)
leak_proba = leaky.predict_proba(df.loc[X_va.index, LEAKY])[:, 1]

print("Honest model,  ROC-AUC:", round(roc_auc_score(ys_va, best_clf.predict_proba(X_va)[:, 1]), 3))
print("Leaky model,   ROC-AUC:", round(roc_auc_score(ys_va, leak_proba), 3))
print()
print("The leaky model is essentially perfect, and completely worthless.")
print("sold_fast is DEFINED as days_on_market < 30. The model has not predicted anything.")
print("It has read the answer off the back of the card.")

No amount of careful cross-validation would have caught this. The leaky model genuinely does
perform that well, on every split, forever. It fails only in production, on the day someone
asks it to score a listing that has not sold yet, and discovers the column is empty.

**The only defence is the question.** For every feature: would I actually have this value at
the moment I need the prediction?

---
# Part 9 - Checkpoint: the honest final number

You have used the validation set many times today: to choose depth, to compare models, to
pick a threshold. It is now contaminated by your choices.

Score your single chosen model on the test set **once**, and report that number. Whatever it
says, that is your answer.

In [ ]:
final_reg = Pipeline([("prep", preprocess), ("model", LinearRegression())])
final_reg.fit(X_tr, np.log(yp_tr))
pred_test = np.exp(final_reg.predict(X_te))

print("=== FINAL, on the test set, reported once ===")
print(f"MAE   {mean_absolute_error(yp_te, pred_test):8.0f} EUR")
print(f"RMSE  {root_mean_squared_error(yp_te, pred_test):8.0f} EUR")
print(f"R2    {r2_score(yp_te, pred_test):8.3f}")
print(f"MAPE  {np.mean(np.abs(pred_test - yp_te) / yp_te) * 100:8.1f} %")
print()
print("Compare against validation. If the test score is much worse, you overfitted your")
print("model SELECTION to the validation set, which is a real and common failure.")

coefs = pd.Series(final_reg.named_steps["model"].coef_,
                  index=final_reg.named_steps["prep"].get_feature_names_out())
print("\nStrongest effects on log price (this is why interpretable models are useful):")
print(coefs.reindex(coefs.abs().sort_values(ascending=False).index).head(8).round(3))

### What you should be able to state before you leave

1. Why stripping whitespace before the split is fine, but filling a median before the split is not.
2. What the gap between the training and validation curves is measuring.
3. The difference between precision and recall, in a sentence, without looking it up.
4. Why the leaky model's perfect ROC-AUC is worthless.
5. Why we scored the test set only once.

---
### Stretch tasks for the afternoon project block

**A. Grid search.** Use `GridSearchCV` to tune the random forest over `max_depth` and
`min_samples_leaf`. Note that it cross-validates inside the training set, so the validation
set stays clean.

**B. Feature importance, carefully.** Compare the random forest's `feature_importances_`
against `sklearn.inspection.permutation_importance`. They disagree, and the reason matters:
the built-in version is biased towards high-cardinality features.

**C. Error analysis.** Find the 20 listings your regression model gets most wrong. Look at
them. Is there a pattern? A group your model systematically fails on is far more useful to
report on Friday than a third decimal place of R squared.

**D. Your project.** Teams form this afternoon. Bring one question about this dataset that you
actually want answered, and a first guess at how you would answer it.